# BELLHOP Underwater Acoustic Propagation Simulator

**Interactive exploration of ray acoustics, transmission loss, and multipath propagation**

This notebook drives [BELLHOP](https://oalib-acoustics.org/) — the industry-standard ray-acoustic
propagation model — through the Python [arlpy](https://github.com/org-arl/arlpy) library.
You can explore how sound propagates through different ocean environments in real time
using interactive sliders and dropdowns.

## What does BELLHOP compute?

| Output | Physical meaning |
|--------|-----------------|
| **Ray paths** | Trajectories of sound energy through the water column — shows refraction, surface/bottom bounces, shadow zones |
| **Transmission Loss (TL)** | How much sound level drops with range and depth — critical for sonar range prediction |
| **Multipath arrivals** | The set of distinct ray paths reaching a receiver — determines channel delay spread for communications |

## Sections in this notebook

1. **Setup** — BELLHOP path config and library imports
2. **Sound Speed Profile Explorer** — Interactive SSP builder with 6 profile types
3. **Ray Tracing** — Visualize ray paths for any source/environment
4. **Transmission Loss Map** — Full 2D TL color plot with convergence zone detection
5. **Multipath Arrivals Analysis** — Delay spread, impulse response, comms implications
6. **Scenario Comparison** — Compare up to 4 environments side-by-side
7. **Sonar Equation Calculator** — Translate TL into SNR for a real sonar system

---


## Section 1 — Setup and Configuration

In [ ]:
%matplotlib inline
# ── Core imports ──────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))   # add project root to path

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── Project modules ───────────────────────────────────────────────────────────
# importlib.reload ensures we always run the on-disk version, not a stale
# bytecode cache from a previous kernel session.
import importlib
import src.bellhop_config as bc
import src.profiles as prof
import src.environment_builder as eb
import src.plotting as pl
import src.utils as ut
for _mod in [bc, prof, eb, pl, ut]:
    importlib.reload(_mod)

# ── Configure BELLHOP ─────────────────────────────────────────────────────────
bc.configure(verbose=True)

# ── Import arlpy AFTER configure() ────────────────────────────────────────────
import arlpy.uwapm as pm

print("\n=== arlpy default environment ===")
pm.print_env(pm.create_env2d())


In [ ]:
def show_fig(fig):
    """Render a matplotlib figure inside an Output widget and close it.

    With %matplotlib inline, plt.show() fires at cell-end, not inside
    widget callbacks. display(fig) explicitly pushes the figure into the
    current Output context, and plt.close() prevents a second render.
    """
    plt.tight_layout()
    display(fig)
    plt.close(fig)


In [ ]:
# ── Quick sanity test (runs a ~1-second BELLHOP job) ─────────────────────────
ok = bc.quick_test()
if not ok:
    print("\n⚠  BELLHOP test failed. Check the path configuration above.")
else:
    print("\n✓  BELLHOP is working correctly — proceed to Section 2.")


---
## Section 2 — Sound Speed Profile Explorer

Select a profile type and adjust its parameters.
The live plot updates as you move the sliders.
Click **Lock in SSP** to save this profile for use in Sections 3–6.

### Physical context
The sound speed profile (SSP) controls everything about how sound propagates:
- **Minimum** speed depth = SOFAR channel axis — sound is trapped here and travels enormous distances
- **Negative gradient** above the axis: sound refracts upward → skip zones, shadow zones
- **Positive gradient** below: sound refracts downward → surface duct or upward refraction


In [ ]:
# ── Shared state ──────────────────────────────────────────────────────────────
_locked_ssp = {'ssp': prof.munk_profile(), 'name': 'Munk (default)'}

# ── Layout helpers ────────────────────────────────────────────────────────────
def _slider(desc, val, lo, hi, step, width='95%'):
    return widgets.FloatSlider(
        value=val, min=lo, max=hi, step=step,
        description=desc, style={'description_width': '170px'},
        layout=widgets.Layout(width=width)
    )

def _int_slider(desc, val, lo, hi, step=1, width='95%'):
    return widgets.IntSlider(
        value=val, min=lo, max=hi, step=step,
        description=desc, style={'description_width': '170px'},
        layout=widgets.Layout(width=width)
    )

# ── Widgets ───────────────────────────────────────────────────────────────────
w_profile = widgets.Dropdown(
    options=['Munk', 'Isothermal', 'Surface Duct', 'Arctic', 'Shallow Water', 'Custom'],
    value='Munk', description='Profile type:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='320px')
)

# Munk sliders
w_c0      = _slider('c₀ (m/s)',      1500, 1400, 1600, 1)
w_z_axis  = _slider('Channel axis (m)', 1300, 200, 3000, 10)
w_epsilon = _slider('ε (strength)',  0.00737, 0.001, 0.02, 0.0001)
w_zmax_m  = _slider('Max depth (m)', 5000, 500, 6000, 50)
munk_box  = widgets.VBox([w_c0, w_z_axis, w_epsilon, w_zmax_m])

# Isothermal
w_iso_speed = _slider('Sound speed (m/s)', 1500, 1400, 1600, 1)
w_iso_zmax  = _slider('Max depth (m)', 3000, 100, 6000, 50)
iso_box = widgets.VBox([w_iso_speed, w_iso_zmax])

# Surface duct
w_duct_depth = _slider('Duct depth (m)', 100, 10, 400, 5)
w_duct_speed = _slider('Duct speed (m/s)', 1520, 1490, 1570, 1)
w_duct_zmax  = _slider('Max depth (m)', 4000, 500, 6000, 50)
duct_box = widgets.VBox([w_duct_depth, w_duct_speed, w_duct_zmax])

# Arctic
w_arctic_surface = _slider('Surface speed (m/s)', 1435, 1400, 1470, 1)
w_arctic_zmax    = _slider('Max depth (m)', 4000, 500, 5000, 50)
arctic_box = widgets.VBox([w_arctic_surface, w_arctic_zmax])

# Shallow water
w_sw_depth    = _slider('Total depth (m)', 200, 30, 500, 5)
w_sw_surf_spd = _slider('Surface speed (m/s)', 1520, 1480, 1560, 1)
w_sw_bot_spd  = _slider('Bottom speed (m/s)', 1490, 1460, 1530, 1)
sw_box = widgets.VBox([w_sw_depth, w_sw_surf_spd, w_sw_bot_spd])

# Custom profile
w_custom_text = widgets.Textarea(
    value='0,1520\n50,1510\n100,1495\n200,1490\n500,1492\n1000,1500\n2000,1510',
    description='depth,speed pairs:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='95%', height='120px')
)
custom_box = widgets.VBox([
    widgets.HTML('<b>Enter one (depth m, speed m/s) pair per line:</b>'),
    w_custom_text
])

# Container that swaps param boxes
params_container = widgets.VBox([munk_box])

ssp_out = widgets.Output()
lock_btn = widgets.Button(description='🔒 Lock in this SSP', button_style='success',
                          layout=widgets.Layout(width='200px'))
ssp_info = widgets.HTML('')

def _get_current_ssp():
    p = w_profile.value
    if p == 'Munk':
        return prof.munk_profile(c0=w_c0.value, z_axis=w_z_axis.value,
                                  epsilon=w_epsilon.value, z_max=w_zmax_m.value)
    elif p == 'Isothermal':
        return prof.isothermal_profile(speed=w_iso_speed.value, z_max=w_iso_zmax.value)
    elif p == 'Surface Duct':
        return prof.surface_duct_profile(duct_depth=w_duct_depth.value,
                                          duct_speed=w_duct_speed.value,
                                          z_max=w_duct_zmax.value)
    elif p == 'Arctic':
        return prof.arctic_profile(surface_speed=w_arctic_surface.value,
                                    z_max=w_arctic_zmax.value)
    elif p == 'Shallow Water':
        return prof.shallow_water_profile(total_depth=w_sw_depth.value,
                                           surface_speed=w_sw_surf_spd.value,
                                           bottom_speed=w_sw_bot_spd.value)
    else:  # Custom
        try:
            pairs = [list(map(float, line.split(',')))
                     for line in w_custom_text.value.strip().split('\n') if line.strip()]
            return prof.custom_profile(pairs)
        except Exception as e:
            return prof.munk_profile()

def _update_ssp(_=None):
    ssp = _get_current_ssp()
    sofar = prof.get_sofar_depth(ssp)
    c_min, c_max = prof.get_speed_range(ssp)
    ssp_info.value = (
        f'<b>SSP stats:</b>  '
        f'Min speed: <b>{c_min:.1f} m/s</b>  |  '
        f'Max speed: <b>{c_max:.1f} m/s</b>  |  '
        f'SOFAR axis: <b>{sofar:.0f} m</b>'
    )
    with ssp_out:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(4.5, 7))
        pl.plot_ssp(ssp, ax=ax, title=f'{w_profile.value} Profile')
        show_fig(fig)

def _on_profile_change(change):
    p = change['new']
    box_map = {
        'Munk': munk_box, 'Isothermal': iso_box,
        'Surface Duct': duct_box, 'Arctic': arctic_box,
        'Shallow Water': sw_box, 'Custom': custom_box
    }
    params_container.children = [box_map.get(p, munk_box)]
    _update_ssp()

def _lock_ssp(_):
    ssp = _get_current_ssp()
    _locked_ssp['ssp'] = ssp
    _locked_ssp['name'] = w_profile.value
    lock_btn.description = f'✓ Locked: {w_profile.value}'
    lock_btn.button_style = 'info'

w_profile.observe(_on_profile_change, names='value')
lock_btn.on_click(_lock_ssp)

# Connect all param sliders to update
for w in [w_c0, w_z_axis, w_epsilon, w_zmax_m,
          w_iso_speed, w_iso_zmax, w_duct_depth, w_duct_speed, w_duct_zmax,
          w_arctic_surface, w_arctic_zmax, w_sw_depth, w_sw_surf_spd, w_sw_bot_spd,
          w_custom_text]:
    w.observe(_update_ssp, names='value')

_update_ssp()

display(widgets.VBox([
    w_profile,
    params_container,
    ssp_info,
    lock_btn,
    ssp_out
]))


---
## Section 3 — Ray Tracing Simulation

Configure the source/receiver geometry and bottom type, then click **Run Ray Trace**.

> **Default source depth = 1300 m** — this is the Munk profile SOFAR channel axis.
> Placing the source at the SOFAR axis produces the classic convergence zone ray fan:
> rays launched at all angles oscillate symmetrically around the axis and focus at
> regular intervals (~65 km for typical deep-ocean profiles).
> Move the source shallower (e.g. 100 m) to see how rays become trapped in the upper
> water column and produce shadow zones below.

### What to look for
- **Convergence zones**: tight bundles of rays focusing at ~65 km intervals — very low TL
- **Shadow zones**: regions where no rays reach — very high TL
- **SOFAR channel**: rays launched at small angles (< 10°) oscillate tightly around the axis
- **Bottom/surface bounces**: steep-angle rays that interact with boundaries lose energy


In [ ]:
# ── Ray trace widgets ─────────────────────────────────────────────────────────
rt_src_depth = _slider('Source depth (m)',  1300, 1, 4999, 1)
rt_water_depth = _slider('Water depth (m)', 5000, 200, 6000, 50)
rt_max_range = _slider('Max range (km)',      100, 1, 500, 1)
rt_n_rays    = _int_slider('Number of rays',  50, 10, 300, 10)
rt_min_angle = _slider('Min launch angle°', -80, -89, 0, 1)
rt_max_angle = _slider('Max launch angle°',  80, 0, 89, 1)
rt_freq      = _slider('Frequency (Hz)',    500, 10, 10000, 10)
rt_bottom    = widgets.Dropdown(
    options=['sand', 'mud', 'gravel', 'rock'],
    value='sand', description='Bottom type:',
    style={'description_width': '120px'}, layout=widgets.Layout(width='300px')
)
rt_run_btn = widgets.Button(description='▶ Run Ray Trace', button_style='primary',
                             layout=widgets.Layout(width='200px'))
rt_out = widgets.Output()
rt_stats = widgets.HTML('')
rt_warn = widgets.HTML('')

_WARN_STYLE = 'color:darkorange;font-weight:bold'

def _rt_check(_=None):
    msgs = []
    sd, wd = rt_src_depth.value, rt_water_depth.value
    if sd >= wd:
        msgs.append(f'⚠ Source depth ({sd:.0f} m) ≥ water depth ({wd:.0f} m) — will be clamped to {wd-1:.0f} m.')
    if rt_min_angle.value >= rt_max_angle.value:
        msgs.append(f'⚠ Min angle ({rt_min_angle.value:.0f}°) ≥ max angle ({rt_max_angle.value:.0f}°) — no rays will be traced.')
    cost = rt_n_rays.value * rt_max_range.value
    if cost > 15000:   # ~150 rays × 100 km or equivalent
        msgs.append(f'⚠ {rt_n_rays.value} rays × {rt_max_range.value:.0f} km — expect a slow run (> 30 s). Reduce rays or range for faster results.')
    rt_warn.value = '<br>'.join(f'<span style="{_WARN_STYLE}">{m}</span>' for m in msgs)

for _w in [rt_src_depth, rt_water_depth, rt_min_angle, rt_max_angle, rt_n_rays, rt_max_range]:
    _w.observe(_rt_check, names='value')
_rt_check()

def _run_ray_trace(_=None):
    if rt_min_angle.value >= rt_max_angle.value:
        with rt_out:
            clear_output(wait=True)
            print('Cannot run: min launch angle must be less than max launch angle.')
        return
    with rt_out:
        clear_output(wait=True)
        print('Running BELLHOP ray trace...')
    water_depth = rt_water_depth.value
    src_depth = min(rt_src_depth.value, water_depth - 1)
    ssp = _locked_ssp['ssp']

    # Clip SSP to water depth
    ssp_use = eb._clip_ssp(ssp, water_depth)

    env = pm.create_env2d(
        depth=water_depth,
        soundspeed=ssp_use.tolist(),
        frequency=rt_freq.value,
        tx_depth=src_depth,
        rx_depth=np.linspace(0, water_depth, 21),
        rx_range=np.linspace(0.1, rt_max_range.value, 25),
        min_angle=rt_min_angle.value,
        max_angle=rt_max_angle.value,
        nbeams=rt_n_rays.value,
    )
    eb._apply_bottom(env, rt_bottom.value)

    rays = pm.compute_rays(env)

    with rt_out:
        clear_output(wait=True)
        if rays is None:
            print('BELLHOP did not produce ray output. Check env parameters.')
            return

        fig, axes = plt.subplots(1, 2, figsize=(16, 5),
                                  gridspec_kw={'width_ratios': [1, 2.5]})

        # Left: SSP
        pl.plot_ssp(ssp_use, ax=axes[0], title=f'SSP: {_locked_ssp["name"]}')

        # Right: ray diagram
        ax_ray = axes[1]
        ax_ray.set_facecolor('#ddeeff')

        max_range_km = rt_max_range.value   # x-axis in km throughout

        # Bottom (x in km, y in m)
        ax_ray.fill_between([0, max_range_km],
                             [water_depth] * 2, [water_depth * 1.08] * 2,
                             color='#8B6914', alpha=0.9, zorder=2)
        ax_ray.axhline(water_depth, color='#8B6914', linewidth=2, zorder=3)
        ax_ray.axhline(0, color='#2196F3', linewidth=2, zorder=3)

        # Plot rays — arlpy stores each path in row['ray'] as (N,2):
        # col 0 = range in km, col 1 = depth in metres
        cmap = plt.cm.RdYlBu
        n_rays = len(rays)
        for i, (_, row) in enumerate(rays.iterrows()):
            if 'ray' in row.index:
                ray_path = np.asarray(row['ray'])
                ax_ray.plot(ray_path[:, 0], ray_path[:, 1],   # col0 already km
                            color=cmap(i / max(n_rays - 1, 1)),
                            linewidth=0.7, alpha=0.65, zorder=4)

        ax_ray.plot(0, src_depth, 'r*', markersize=16, zorder=10,
                    label=f'Source: {src_depth:.0f} m')
        ax_ray.set_xlim(0, max_range_km)
        ax_ray.set_ylim(water_depth * 1.08, -water_depth * 0.02)
        ax_ray.set_xlabel('Range (km)', fontsize=11)
        ax_ray.set_ylabel('Depth (m)', fontsize=11)
        ax_ray.set_title(
            f'Ray Diagram  |  f={rt_freq.value:.0f} Hz  |  {n_rays} rays  |  '
            f'{rt_bottom.value} bottom  |  SSP: {_locked_ssp["name"]}',
            fontsize=12)
        ax_ray.grid(True, alpha=0.2)
        ax_ray.legend(fontsize=9)

        show_fig(fig)

        rt_stats.value = (
            f'<b>Ray statistics:</b>  '
            f'Total rays launched: <b>{n_rays}</b>  |  '
            f'Env: {rt_bottom.value} bottom, {water_depth:.0f} m deep, '
            f'src @ {src_depth:.0f} m, max range {rt_max_range.value:.0f} km'
        )

rt_run_btn.on_click(_run_ray_trace)

display(widgets.VBox([
    widgets.HBox([rt_src_depth, rt_water_depth]),
    widgets.HBox([rt_max_range, rt_n_rays]),
    widgets.HBox([rt_min_angle, rt_max_angle]),
    widgets.HBox([rt_freq, rt_bottom]),
    rt_warn,
    rt_run_btn,
    rt_stats,
    rt_out
]))


---
## Section 4 — Transmission Loss Map

Computes and displays the full 2D transmission loss field (depth × range).

TL is computed in dB relative to 1 m from the source:
$$TL = -20\log_{10}\left(\frac{p(r,z)}{p_0}\right)$$

Higher TL = weaker signal. Convergence zones appear as vertical bands of low TL.


In [ ]:
# ── TL shared state ──────────────────────────────────────────────────────────
_tl_result = {'tl': None, 'env': None}
tl_ready = widgets.HTML(
    '<span style="color:gray">No TL data yet — click <b>▶ Run TL Map</b> above.</span>'
)

# ── TL widgets ────────────────────────────────────────────────────────────────
tl_src_depth   = _slider('Source depth (m)',   1300, 1, 4999, 1)
tl_water_depth = _slider('Water depth (m)',    5000, 200, 6000, 50)
tl_max_range   = _slider('Max range (km)',      100, 1, 500, 1)
tl_freq        = _slider('Frequency (Hz)',      500, 10, 10000, 10)
tl_rx_min_d    = _slider('Rx min depth (m)',      0, 0, 4900, 10)
tl_rx_max_d    = _slider('Rx max depth (m)',   4950, 100, 5000, 10)
tl_n_depths    = _int_slider('# Rx depths',      51, 5, 201, 5)
tl_dyn_range   = _slider('Dynamic range (dB)',   60, 20, 100, 5)
tl_bottom      = widgets.Dropdown(
    options=['sand', 'mud', 'gravel', 'rock'],
    value='sand', description='Bottom type:',
    style={'description_width': '120px'}, layout=widgets.Layout(width='300px')
)
tl_run_type = widgets.Dropdown(
    options=['coherent', 'incoherent', 'semicoherent'],
    value='incoherent', description='Run type:',
    style={'description_width': '120px'}, layout=widgets.Layout(width='300px')
)
tl_slice_depth = _slider('TL slice depth (m)', 1300, 0, 5000, 10)
tl_run_btn = widgets.Button(description='▶ Run TL Map', button_style='primary',
                              layout=widgets.Layout(width='200px'))
tl_out    = widgets.Output()
tl_stats  = widgets.HTML('')
tl_slice_btn = widgets.Button(description='Plot TL slice', button_style='',
                               layout=widgets.Layout(width='160px'))
tl_slice_out = widgets.Output()
tl_warn = widgets.HTML('')

def _tl_check(_=None):
    msgs = []
    sd, wd = tl_src_depth.value, tl_water_depth.value
    rmin, rmax = tl_rx_min_d.value, tl_rx_max_d.value
    if sd >= wd:
        msgs.append(f'⚠ Source depth ({sd:.0f} m) ≥ water depth ({wd:.0f} m) — will be clamped to {wd-1:.0f} m.')
    if rmin >= rmax:
        msgs.append(f'⚠ Rx min depth ({rmin:.0f} m) ≥ Rx max depth ({rmax:.0f} m) — no receiver grid possible.')
    if rmax > wd:
        msgs.append(f'⚠ Rx max depth ({rmax:.0f} m) exceeds water depth ({wd:.0f} m) — will be clamped to {wd:.0f} m.')
    sd_tl = tl_slice_depth.value
    if sd_tl < rmin or sd_tl > rmax:
        msgs.append(f'⚠ Slice depth ({sd_tl:.0f} m) is outside the receiver depth range ({rmin:.0f}–{rmax:.0f} m) — slice will show nearest available depth.')
    tl_warn.value = '<br>'.join(f'<span style="{_WARN_STYLE}">{m}</span>' for m in msgs)

for _w in [tl_src_depth, tl_water_depth, tl_rx_min_d, tl_rx_max_d, tl_slice_depth]:
    _w.observe(_tl_check, names='value')
_tl_check()

run_type_map = {
    'coherent': pm.coherent,
    'incoherent': pm.incoherent,
    'semicoherent': pm.semicoherent,
}

def _run_tl(_=None):
    if tl_rx_min_d.value >= tl_rx_max_d.value:
        with tl_out:
            clear_output(wait=True)
            print('Cannot run: Rx min depth must be less than Rx max depth.')
        return
    with tl_out:
        clear_output(wait=True)
        print('Running BELLHOP transmission loss computation...')
    water_depth = tl_water_depth.value
    src_depth = min(tl_src_depth.value, water_depth - 1)
    rx_min = min(tl_rx_min_d.value, water_depth)
    rx_max = min(tl_rx_max_d.value, water_depth)
    ssp = _locked_ssp['ssp']
    ssp_use = eb._clip_ssp(ssp, water_depth)

    env = pm.create_env2d(
        depth=water_depth,
        soundspeed=ssp_use.tolist(),
        frequency=tl_freq.value,
        tx_depth=src_depth,
        rx_depth=np.linspace(rx_min, rx_max, int(tl_n_depths.value)),
        rx_range=np.linspace(0.5, tl_max_range.value, 100),
        nbeams=200,
    )
    eb._apply_bottom(env, tl_bottom.value)

    task = run_type_map.get(tl_run_type.value, pm.incoherent)

    try:
        tl = pm.compute_transmission_loss(env, mode=task)
    except Exception as exc:
        import traceback
        with tl_out:
            clear_output(wait=True)
            print(f'BELLHOP error: {type(exc).__name__}: {exc}')
            traceback.print_exc()
        tl_ready.value = '<span style="color:red">✗ TL computation failed — see output above.</span>'
        return

    _tl_result['tl'] = tl
    _tl_result['env'] = env

    with tl_out:
        clear_output(wait=True)
        if tl is None:
            print('BELLHOP did not produce TL output.')
            tl_ready.value = '<span style="color:red">✗ TL returned None.</span>'
            return

        try:
            fig, axes = plt.subplots(1, 2, figsize=(16, 5),
                                      gridspec_kw={'width_ratios': [1, 3.5]})

            pl.plot_ssp(ssp_use, ax=axes[0], title=f'SSP: {_locked_ssp["name"]}')
            _, cb = pl.plot_transmission_loss(
                tl, env, ax=axes[1],
                dynamic_range=tl_dyn_range.value,
                title=f'{tl_run_type.value.title()} TL  |  '
                      f'f={tl_freq.value:.0f} Hz  |  {tl_bottom.value} bottom'
            )

            # Convergence zone detection
            cz_ranges = ut.find_convergence_zones(tl, threshold_db=4.0)
            if cz_ranges:
                pl.annotate_convergence_zones(axes[1], cz_ranges)
                cz_str = ', '.join(f'{r:.0f} km' for r in cz_ranges)
                tl_stats.value = (
                    f'<b>Convergence zones detected at:</b> {cz_str}'
                    f' | Run type: {tl_run_type.value}'
                )
            else:
                tl_stats.value = (
                    f'<b>No convergence zones detected</b> | Run type: {tl_run_type.value}'
                )

            show_fig(fig)

            tl_ready.value = (
                f'<span style="color:green;font-weight:bold">✓ TL ready</span> — '
                f'{tl_run_type.value}, f={tl_freq.value:.0f} Hz, '
                f'src={src_depth:.0f} m, max {tl_max_range.value:.0f} km. '
                f'Use <b>Pull TL from Section 4</b> in Section 7.'
            )

        except Exception as plot_exc:
            import traceback
            print(f'Plot error: {type(plot_exc).__name__}: {plot_exc}')
            traceback.print_exc()
            tl_ready.value = '<span style="color:orange">⚠ TL computed but plot failed.</span>'

def _plot_tl_slice(_=None):
    tl = _tl_result.get('tl')
    env = _tl_result.get('env')
    if tl is None:
        with tl_slice_out:
            clear_output(wait=True)
            print('Run the TL map first.')
        return
    with tl_slice_out:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(11, 4))
        pl.plot_tl_slice(tl, tl_slice_depth.value, env, ax=ax)
        show_fig(fig)

tl_run_btn.on_click(_run_tl)
tl_slice_btn.on_click(_plot_tl_slice)

display(widgets.VBox([
    widgets.HBox([tl_src_depth, tl_water_depth]),
    widgets.HBox([tl_max_range, tl_freq]),
    widgets.HBox([tl_rx_min_d, tl_rx_max_d, tl_n_depths]),
    widgets.HBox([tl_dyn_range, tl_bottom, tl_run_type]),
    tl_warn,
    tl_run_btn,
    tl_ready,
    tl_stats,
    tl_out,
    widgets.HTML('<hr><b>Horizontal TL slice</b>'),
    widgets.HBox([tl_slice_depth, tl_slice_btn]),
    tl_slice_out,
]))


---
## Section 5 — Multipath Arrivals Analysis

BELLHOP traces every ray path from source to receiver and records:
- **Arrival time** — when each ray path reaches the receiver
- **Complex amplitude** — amplitude and phase of each path

### Why does this matter for communications?
In an underwater acoustic channel, every symbol you transmit arrives multiple times via
different paths. Each copy is delayed and scaled differently — this is **multipath**.

The **delay spread** $T_d$ (latest minus earliest arrival) determines:
- **Coherence bandwidth**: $B_c \approx 1/T_d$ — signals wider than this suffer ISI
- **ISI length**: $L_{ISI} = \lfloor T_d \cdot f_s \rfloor$ taps — the equalizer must handle this many

For typical ocean channels: $T_d \approx 10$–$200$ ms → $B_c \approx 5$–$100$ Hz
(Compare: OFDM subcarrier spacing must be $\gg B_c$)

### Getting interesting multipath

The number of arrivals depends strongly on geometry:

| Setting | Result |
|---------|--------|
| Source + Rx both on SOFAR axis (1300 m), range < 65 km | **1 arrival** — only the direct horizontal path returns to that depth; oscillating rays are mid-cycle |
| Source on SOFAR axis, range ≈ **65 km** (1st convergence zone) | **Many arrivals** — all oscillating rays refocus at the CZ |
| Source on SOFAR axis, Rx at shallow depth (e.g. 100 m) | **Several arrivals** — catches the upward-swinging rays mid-cycle |

> **Default is 65 km** — the first convergence zone of the Munk profile where multipath is richest.


In [ ]:
# ── Arrivals shared state ────────────────────────────────────────────────────
_arr_result = {'arr': None, 'env': None}

# ── Arrivals widgets ──────────────────────────────────────────────────────────
arr_src_depth   = _slider('Source depth (m)', 1300, 1, 4999, 1)
arr_water_depth = _slider('Water depth (m)', 5000, 200, 6000, 50)
arr_freq        = _slider('Frequency (Hz)',   500, 10, 5000, 10)
arr_bottom      = widgets.Dropdown(
    options=['sand', 'mud', 'gravel', 'rock'],
    value='sand', description='Bottom type:',
    style={'description_width': '120px'}, layout=widgets.Layout(width='300px')
)
arr_rx_range = _slider('Rx range (km)',  65, 0.5, 300, 0.5)
arr_rx_depth = _slider('Rx depth (m)', 1300, 1, 4999, 1)
arr_fs       = _slider('Sample rate fs (Hz)', 8000, 1000, 50000, 500)
arr_run_btn  = widgets.Button(description='▶ Compute Arrivals', button_style='primary',
                               layout=widgets.Layout(width='220px'))
arr_out      = widgets.Output()
arr_stats    = widgets.HTML('')
arr_warn     = widgets.HTML('')

def _arr_check(_=None):
    msgs = []
    sd, wd = arr_src_depth.value, arr_water_depth.value
    rd, rng = arr_rx_depth.value, arr_rx_range.value
    if sd >= wd:
        msgs.append(f'⚠ Source depth ({sd:.0f} m) ≥ water depth ({wd:.0f} m) — will be clamped to {wd-1:.0f} m.')
    if rd >= wd:
        msgs.append(f'⚠ Rx depth ({rd:.0f} m) ≥ water depth ({wd:.0f} m) — will be clamped to {wd-1:.0f} m.')
    if abs(rd - sd) > 500 and rng < 20:
        msgs.append(f'⚠ Source ({sd:.0f} m) and Rx ({rd:.0f} m) are far apart in depth at only {rng:.0f} km range — few or no arrivals likely. Try range > 20 km or match source/Rx depths.')
    if abs(rd - sd) < 200 and rng < 60:
        msgs.append(f'ℹ Source and Rx near same depth ({sd:.0f} m) at {rng:.0f} km — SOFAR cycle ≈ 65 km, so most oscillating rays miss this receiver. Try range ≈ 65 km for the first convergence zone (many arrivals).')
    arr_warn.value = '<br>'.join(f'<span style="{_WARN_STYLE}">{m}</span>' for m in msgs)

for _w in [arr_src_depth, arr_water_depth, arr_rx_depth, arr_rx_range]:
    _w.observe(_arr_check, names='value')
_arr_check()

def _run_arrivals(_=None):
    water_depth = arr_water_depth.value
    src_depth = min(arr_src_depth.value, water_depth - 1)
    rx_depth  = min(arr_rx_depth.value,  water_depth - 1)
    ssp_use   = eb._clip_ssp(_locked_ssp['ssp'], water_depth)

    env = pm.create_env2d(
        depth=water_depth,
        soundspeed=ssp_use.tolist(),
        frequency=arr_freq.value,
        tx_depth=src_depth,
        rx_depth=np.array([rx_depth]),
        rx_range=np.array([arr_rx_range.value]),
        nbeams=200,
    )
    eb._apply_bottom(env, arr_bottom.value)

    with arr_out:
        clear_output(wait=True)
        print(f'Running BELLHOP arrivals — src={src_depth:.0f} m  rx={rx_depth:.0f} m @ {arr_rx_range.value:.1f} km ...')

    try:
        arr = pm.compute_arrivals(env)
    except ValueError:
        # pandas ≥2.0: pd.concat([]) raises ValueError when BELLHOP finds no arrivals
        with arr_out:
            clear_output(wait=True)
            print('No arrivals found (BELLHOP returned empty result).')
            print('Suggestions:')
            print('  • Increase range (try ≥ 50 km for deep water Munk profile)')
            print('  • Match Rx depth to source depth for maximum energy coupling')
            print('  • Verify source depth < water depth')
        return
    except Exception as exc:
        import traceback
        with arr_out:
            clear_output(wait=True)
            print(f'BELLHOP error: {type(exc).__name__}: {exc}')
            traceback.print_exc()
        return

    _arr_result['arr'] = arr
    _arr_result['env'] = env

    with arr_out:
        clear_output(wait=True)
        try:
            if arr is None or len(arr) == 0:
                print('No arrivals returned (empty DataFrame).')
                print('Try a longer range or different geometry.')
                return

            _amp_col = 'arrival_amplitude' if 'arrival_amplitude' in arr.columns else 'amplitude'
            if 'time_of_arrival' not in arr.columns or _amp_col not in arr.columns:
                print(f'Unexpected arrival columns: {list(arr.columns)}')
                print('Expected: time_of_arrival, arrival_amplitude (or amplitude)')
                return

            times_s = np.asarray(arr['time_of_arrival'], dtype=float)
            amps    = np.abs(np.asarray(arr[_amp_col], dtype=complex))
            valid   = np.isfinite(times_s) & np.isfinite(amps) & (amps > 0)
            times_s = times_s[valid]
            amps    = amps[valid]

            if len(times_s) == 0:
                print(f'All {len(arr)} arrivals have NaN or zero amplitude.')
                print('BELLHOP may have found eigenrays but with negligible energy.')
                print('Try: match Rx depth to source depth, or increase range.')
                return

            delay_ms = (times_s - times_s.min()) * 1000.0
            amps_n   = amps / amps.max()
            spread   = float(delay_ms.max())

            if spread > 0:
                arr_stats.value = (
                    f'<b>Arrivals:</b> {len(times_s)} paths  |  '
                    f'Delay spread: <b>{spread:.2f} ms</b>  |  '
                    f'Coherence BW: <b>{1000.0 / spread:.1f} Hz</b>'
                )
            else:
                arr_stats.value = (
                    f'<b>Arrivals:</b> {len(times_s)} paths  |  '
                    f'Single eigenray (0 ms spread)'
                )

            # ── Bounce-count colors ───────────────────────────────────────────
            _bp = ['#1a6fa3', '#27ae60', '#e67e22', '#c0392b']
            if 'surface_bounces' in arr.columns and 'bottom_bounces' in arr.columns:
                _n_tot = (np.asarray(arr['surface_bounces'], dtype=int) +
                          np.asarray(arr['bottom_bounces'],  dtype=int))[valid]
                _colors = [_bp[min(int(n), 3)] for n in _n_tot]
            else:
                _colors = [_bp[0]] * len(delay_ms)

            fig, axes = plt.subplots(1, 2, figsize=(16, 4))

            # ── Left: multipath arrivals ──────────────────────────────────────
            x_span = max(spread, 1.0)
            axes[0].vlines(delay_ms, 0, amps_n, colors=_colors, linewidth=2, alpha=0.85)
            axes[0].scatter(delay_ms, amps_n, c=_colors, s=60, zorder=5)
            axes[0].axhline(0, color='k', linewidth=0.8)
            axes[0].set_xlim(-x_span * 0.05, x_span * 1.15)
            axes[0].set_ylim(-0.05, 1.15)
            axes[0].set_xlabel('Delay relative to first arrival (ms)', fontsize=11)
            axes[0].set_ylabel('Normalized Amplitude', fontsize=11)
            axes[0].set_title(
                f'Multipath Arrivals  |  {len(times_s)} paths  |  '
                f'spread={spread:.2f} ms  |  '
                f'f={arr_freq.value:.0f} Hz  |  '
                f'Rx {arr_rx_range.value:.0f} km / {rx_depth:.0f} m',
                fontsize=10
            )
            axes[0].grid(True, alpha=0.25)
            # Bounce legend
            from matplotlib.lines import Line2D as _L2D
            _leg = [_L2D([0],[0], color=c, lw=2, label=lbl)
                    for c, lbl in zip(_bp, ['0 bounces', '1 bounce', '2 bounces', '3+ bounces'])]
            axes[0].legend(handles=_leg, fontsize=8, loc='upper right')

            # ── Right: channel impulse response ───────────────────────────────
            fs      = arr_fs.value
            ir      = ut.channel_to_impulse_response(arr, fs=fs)
            t_ms    = np.arange(len(ir)) / fs * 1000.0
            ir_abs  = np.abs(ir)
            ir_max  = float(ir_abs.max()) if ir_abs.max() > 0 else 1.0
            ir_norm = ir_abs / ir_max      # normalize: weak arrivals visible
            sig     = ir_norm > 0
            if sig.any():
                axes[1].vlines(t_ms[sig], 0, ir_norm[sig],
                               colors='#e67e22', linewidth=2, alpha=0.85)
                axes[1].scatter(t_ms[sig], ir_norm[sig], color='#e67e22', s=50, zorder=5)
                x2_span = max(float(t_ms[sig].max()), 1.0)
                axes[1].set_xlim(-x2_span * 0.05, x2_span * 1.15)
            axes[1].axhline(0, color='k', linewidth=0.8)
            axes[1].set_ylim(-0.05, 1.15)
            stats_val = ut.compute_arrival_stats(arr)
            isi_taps  = int(stats_val.get('delay_spread_ms', 0) / 1000.0 * fs)
            axes[1].set_xlabel('Delay (ms)', fontsize=11)
            axes[1].set_ylabel('Normalized Amplitude', fontsize=11)
            axes[1].set_title(
                f'Channel Impulse Response  |  fs={fs:.0f} Hz  |  ISI taps ≈ {isi_taps}',
                fontsize=11
            )
            axes[1].grid(True, alpha=0.25)

            show_fig(fig)

        except Exception as _plot_err:
            import traceback
            print(f'Plotting error: {type(_plot_err).__name__}: {_plot_err}')
            print()
            print('--- Diagnostic ---')
            if arr is not None and len(arr) > 0:
                print(f'Columns : {list(arr.columns)}')
                print(f'Rows    : {len(arr)}')
                _ac = 'arrival_amplitude' if 'arrival_amplitude' in arr.columns else 'amplitude'
                if 'time_of_arrival' in arr.columns and _ac in arr.columns:
                    _t = np.asarray(arr['time_of_arrival'], dtype=float)
                    _a = np.abs(np.asarray(arr[_ac], dtype=complex))
                    _v = np.isfinite(_t) & np.isfinite(_a) & (_a > 0)
                    print(f'Valid   : {int(_v.sum())}/{len(arr)}')
                    if _v.sum() > 0:
                        print(f'Time    : {_t[_v].min():.4f}–{_t[_v].max():.4f} s '
                              f'(spread={1000*(_t[_v].max()-_t[_v].min()):.2f} ms)')
                        print(f'Amp     : {_a[_v].min():.3e}–{_a[_v].max():.3e}')
            traceback.print_exc()

arr_run_btn.on_click(_run_arrivals)

display(widgets.VBox([
    widgets.HBox([arr_src_depth, arr_water_depth]),
    widgets.HBox([arr_freq, arr_bottom]),
    widgets.HBox([arr_rx_range, arr_rx_depth]),
    arr_fs,
    arr_warn,
    arr_run_btn,
    arr_stats,
    arr_out,
]))


---
## Section 6 — Scenario Comparison

Save up to 4 named scenarios (each with its own SSP, depth, frequency, bottom) then
compare their TL-vs-range curves and summary statistics side-by-side.

**Workflow:**
1. Configure the SSP in Section 2 and lock it
2. Set the parameters below and click **Save Scenario**
3. Repeat for up to 4 scenarios
4. Click **Compare Scenarios** to overlay the TL curves


In [ ]:
# ── Scenario storage ──────────────────────────────────────────────────────────
_scenarios = []   # list of dicts: {name, ssp, ssp_name, water_depth, src_depth, freq, bottom}

# ── Scenario widgets ──────────────────────────────────────────────────────────
sc_name       = widgets.Text(value='Scenario 1', description='Name:',
                              style={'description_width': '80px'},
                              layout=widgets.Layout(width='280px'))
sc_water_d    = _slider('Water depth (m)',  5000, 200, 6000, 50)
sc_src_d      = _slider('Source depth (m)', 1300, 1, 4999, 1)
sc_freq       = _slider('Frequency (Hz)',    500, 10, 5000, 10)
sc_bottom     = widgets.Dropdown(
    options=['sand', 'mud', 'gravel', 'rock'],
    value='sand', description='Bottom:',
    style={'description_width': '80px'}, layout=widgets.Layout(width='260px')
)
sc_max_range  = _slider('Max range (km)',   100, 5, 500, 5)
sc_save_btn   = widgets.Button(description='💾 Save Scenario', button_style='warning',
                                layout=widgets.Layout(width='200px'))
sc_compare_btn= widgets.Button(description='📊 Compare Scenarios', button_style='success',
                                layout=widgets.Layout(width='220px'))
sc_clear_btn  = widgets.Button(description='🗑 Clear All', button_style='danger',
                                layout=widgets.Layout(width='150px'))
sc_list_html  = widgets.HTML('<i>No scenarios saved yet.</i>')
sc_warn       = widgets.HTML('')
sc_out        = widgets.Output()

def _sc_check(_=None):
    msgs = []
    if sc_src_d.value >= sc_water_d.value:
        msgs.append(f'⚠ Source depth ({sc_src_d.value:.0f} m) ≥ water depth ({sc_water_d.value:.0f} m) — will be clamped on save.')
    sc_warn.value = '<br>'.join(f'<span style="{_WARN_STYLE}">{m}</span>' for m in msgs)

for _w in [sc_src_d, sc_water_d]:
    _w.observe(_sc_check, names='value')
_sc_check()

def _update_sc_list():
    if not _scenarios:
        sc_list_html.value = '<i>No scenarios saved yet.</i>'
        return
    rows = '<br>'.join(
        f'<b>{i+1}. {s["name"]}</b>: SSP={s["ssp_name"]}, '
        f'{s["water_depth"]:.0f}m deep, src@{s["src_depth"]:.0f}m, '
        f'f={s["freq"]:.0f}Hz, {s["bottom"]}, max {s["max_range"]:.0f}km'
        for i, s in enumerate(_scenarios)
    )
    sc_list_html.value = rows

def _save_scenario(_=None):
    if len(_scenarios) >= 4:
        sc_list_html.value = '<b style="color:red">Maximum 4 scenarios already saved. Clear to start over.</b>'
        return
    _scenarios.append({
        'name':        sc_name.value or f'Scenario {len(_scenarios)+1}',
        'ssp':         _locked_ssp['ssp'].copy(),
        'ssp_name':    _locked_ssp['name'],
        'water_depth': sc_water_d.value,
        'src_depth':   min(sc_src_d.value, sc_water_d.value - 1),
        'freq':        sc_freq.value,
        'bottom':      sc_bottom.value,
        'max_range':   sc_max_range.value,
    })
    sc_name.value = f'Scenario {len(_scenarios)+1}'
    _update_sc_list()

def _compare_scenarios(_=None):
    with sc_out:
        clear_output(wait=True)
        if len(_scenarios) < 1:
            print('Save at least one scenario first.')
            return
        print(f'Running TL for {len(_scenarios)} scenarios...')

        tl_results, envs, labels = [], [], []
        for s in _scenarios:
            ssp_use = eb._clip_ssp(s['ssp'], s['water_depth'])
            env = pm.create_env2d(
                depth=s['water_depth'],
                soundspeed=ssp_use.tolist(),
                frequency=s['freq'],
                tx_depth=s['src_depth'],
                rx_depth=np.linspace(0, s['water_depth'], 21),
                rx_range=np.linspace(0.5, s['max_range'], 80),
                nbeams=200,
            )
            eb._apply_bottom(env, s['bottom'])
            tl = pm.compute_transmission_loss(env, mode=pm.incoherent)
            tl_results.append(tl)
            envs.append(env)
            labels.append(s['name'])
            print(f'  Completed: {s["name"]}')

        # Comparison TL plot
        fig, axes = plt.subplots(1, 2, figsize=(16, 5))

        colors = plt.cm.tab10(np.linspace(0, 0.9, len(_scenarios)))
        for i, (tl, env, label, s) in enumerate(zip(tl_results, envs, labels, _scenarios)):
            if tl is None:
                continue
            depths = np.asarray(tl.index, dtype=float)
            ranges_km = np.asarray(tl.columns, dtype=float)
            d_ref = s['src_depth']
            d_idx = int(np.argmin(np.abs(depths - d_ref)))
            p_abs = np.abs(np.asarray(tl.iloc[d_idx, :], dtype=complex))
            p_abs = np.where(p_abs < 1e-10, np.nan, p_abs)
            tl_slice = -20.0 * np.log10(p_abs)
            axes[0].plot(ranges_km, tl_slice, color=colors[i], linewidth=2, label=label)

        # SSP overlay
        for i, s in enumerate(_scenarios):
            axes[1].plot(s['ssp'][:, 1], s['ssp'][:, 0],
                         color=colors[i], linewidth=2, label=s['name'])

        axes[0].invert_yaxis()
        axes[0].set_xlabel('Range (km)', fontsize=11)
        axes[0].set_ylabel('Transmission Loss (dB)', fontsize=11)
        axes[0].set_title('TL vs Range at Source Depth', fontsize=12)
        axes[0].grid(True, alpha=0.25)
        axes[0].legend(fontsize=9)

        axes[1].invert_yaxis()
        axes[1].set_xlabel('Sound Speed (m/s)', fontsize=11)
        axes[1].set_ylabel('Depth (m)', fontsize=11)
        axes[1].set_title('Sound Speed Profiles', fontsize=12)
        axes[1].grid(True, alpha=0.25)
        axes[1].legend(fontsize=9)

        plt.suptitle('Scenario Comparison', fontsize=14, fontweight='bold')
        show_fig(fig)

        # Summary table
        import pandas as pd
        rows = []
        for s in _scenarios:
            rows.append({
                'Name': s['name'], 'SSP': s['ssp_name'],
                'Depth (m)': s['water_depth'], 'Src (m)': s['src_depth'],
                'Freq (Hz)': s['freq'], 'Bottom': s['bottom'],
                'Max Range (km)': s['max_range'],
            })
        df = pd.DataFrame(rows)
        from IPython.display import display as ipy_display
        ipy_display(df)

def _clear_scenarios(_=None):
    _scenarios.clear()
    _update_sc_list()
    with sc_out:
        clear_output(wait=True)

sc_save_btn.on_click(_save_scenario)
sc_compare_btn.on_click(_compare_scenarios)
sc_clear_btn.on_click(_clear_scenarios)

display(widgets.VBox([
    widgets.HBox([sc_name, sc_bottom]),
    widgets.HBox([sc_water_d, sc_src_d]),
    widgets.HBox([sc_freq, sc_max_range]),
    sc_warn,
    widgets.HBox([sc_save_btn, sc_compare_btn, sc_clear_btn]),
    sc_list_html,
    sc_out,
]))


---
## Section 7 — Sonar Equation Calculator

The passive sonar equation:

$$SNR = SL - TL - NL + DI \quad \text{(all in dB)}$$

| Term | Symbol | Meaning |
|------|--------|---------|
| Source Level | SL | Radiated power of target (dB re 1 µPa @ 1 m) |
| Transmission Loss | TL | Acoustic path loss (from Section 4 or manual) |
| Noise Level | NL | Ambient ocean noise at receiver (dB re 1 µPa) |
| Directivity Index | DI | Receiver array gain over an omnidirectional sensor (dB) |
| Detection Threshold | DT | Min SNR for reliable detection (dB) |

A target is **detectable** when $SNR \geq DT$.


In [ ]:
# ── Sonar equation widgets ────────────────────────────────────────────────────
sn_sl  = _slider('SL — Source Level (dB)',      200, 140, 240, 1)
sn_nl  = _slider('NL — Noise Level (dB)',        60, 30, 100, 1)
sn_di  = _slider('DI — Directivity Index (dB)',   0, 0, 40, 1)
sn_dt  = _slider('DT — Detection Threshold (dB)',  10, 0, 30, 1)
sn_tl_manual = _slider('TL (dB)', 80, 20, 150, 0.1)
sn_tl_range  = _slider('Lookup range (km)',  50, 0.5, 500, 0.5)
sn_tl_depth  = _slider('Lookup depth (m)', 1300, 0, 5000, 10)
sn_tl_from4  = widgets.Button(
    description='Pull TL from Section 4',
    button_style='info', layout=widgets.Layout(width='220px')
)
sn_pull_status = widgets.HTML(
    '<i style="color:gray">Set the lookup range/depth above, then click the button '
    'to read TL at that point from the Section 4 map.</i>'
)
sn_out = widgets.Output()

_sonar_state = {'tl': 80.0}

def _update_sonar(_=None):
    tl = _sonar_state['tl']
    snr = ut.compute_snr_estimate(tl, sn_sl.value, sn_nl.value, sn_di.value)
    margin = snr - sn_dt.value
    verdict = 'DETECTED ✓' if margin >= 0 else 'NOT DETECTED ✗'
    with sn_out:
        clear_output(wait=True)
        print(f'  SL  = {sn_sl.value:.0f} dB')
        print(f'  TL  = {tl:.1f} dB')
        print(f'  NL  = {sn_nl.value:.0f} dB')
        print(f'  DI  = {sn_di.value:.0f} dB')
        print(f'─' * 30)
        print(f'  SNR = {snr:.1f} dB')
        print(f'  DT  = {sn_dt.value:.0f} dB')
        print(f'  Margin = {margin:+.1f} dB  →  {verdict}')

def _pull_tl_from_sec4(_=None):
    tl_df = _tl_result.get('tl')
    if tl_df is None:
        sn_pull_status.value = (
            '<span style="color:red;font-weight:bold">'
            '⚠ No TL data found. In Section 4: set parameters, then click '
            '<b>▶ Run TL Map</b>. The status line will turn green when ready.</span>'
        )
        return

    depths  = np.asarray(tl_df.index,   dtype=float)
    ranges  = np.asarray(tl_df.columns, dtype=float)
    d_idx   = int(np.argmin(np.abs(depths - sn_tl_depth.value)))
    r_idx   = int(np.argmin(np.abs(ranges - sn_tl_range.value)))
    p_val   = float(np.abs(complex(tl_df.iloc[d_idx, r_idx])))
    tl_val  = float(-20.0 * np.log10(p_val)) if p_val > 1e-10 else 120.0

    # Clamp to slider range before setting (avoids silent widget clamp)
    tl_clamped = float(np.clip(tl_val, 20, 150))
    _sonar_state['tl'] = tl_clamped
    sn_tl_manual.value  = tl_clamped

    actual_r = float(ranges[r_idx])
    actual_d = float(depths[d_idx])
    sn_pull_status.value = (
        f'<span style="color:green;font-weight:bold">'
        f'✓ Pulled TL = {tl_val:.1f} dB '
        f'from Section 4 at {actual_r:.1f} km / {actual_d:.0f} m depth</span>'
        + (f'<br><span style="color:gray;font-size:0.85em">'
           f'(Nearest computed point to your lookup sliders)</span>'
           if abs(actual_r - sn_tl_range.value) > 1 or abs(actual_d - sn_tl_depth.value) > 50
           else '')
    )
    _update_sonar()

def _manual_tl_changed(change):
    _sonar_state['tl'] = change['new']
    sn_pull_status.value = '<i style="color:gray">TL set manually.</i>'
    _update_sonar()

for w in [sn_sl, sn_nl, sn_di, sn_dt]:
    w.observe(_update_sonar, names='value')
sn_tl_manual.observe(_manual_tl_changed, names='value')
sn_tl_from4.on_click(_pull_tl_from_sec4)

_update_sonar()

display(widgets.VBox([
    widgets.HBox([sn_sl, sn_nl]),
    widgets.HBox([sn_di, sn_dt]),
    widgets.HTML('<hr><b>Transmission Loss (TL)</b><br>'
                 '<span style="color:gray;font-size:0.9em">'
                 'Either drag the slider to enter TL manually, or use the Section 4 lookup below.</span>'),
    sn_tl_manual,
    widgets.HTML('<b>Section 4 TL lookup</b> — set range &amp; depth, then click the button:'),
    widgets.HBox([sn_tl_range, sn_tl_depth]),
    widgets.HBox([sn_tl_from4, sn_pull_status]),
    widgets.HTML('<hr>'),
    sn_out,
]))
